# FORMATIVE ASSESSMENT OF ADOLESCENT GIRLS AND YOUNG WOMEN’S HIV, GENDER-BASED VIOLENCE AND SEXUAL AND REPRODUCTIVE HEALTH STATUS

## Background
Teenage pregnancy and motherhood have been a major health and social concern in Uganda as it infringes upon the human rights of girls but also hinders their ability to achieve their full socioeconomic development. Teenagers who engage in sexual intercourse at a young age face an elevated risk of becoming pregnant and giving birth. The 2022 UDHS indicated that 23.5% of women age 15-19 had initiated childbearing by the time of the survey, with 18.4% having already had a live birth, while 5.1% were pregnant with their first child.

Patterns by background characteristics:
* By age 16, 1 in every 10 women age 15-19 has begun childbearing. This percentage significantly rises to almost 4 out of every 10 by the time they reach 18.
* Teenagers in rural areas started childbearing earlier than those in urban areas. Twenty five percent of women age 15-19 in rural areas have begun childbearing, compared with 21% in urban areas.
* Teenage childbearing varies by region. The percentage of women age 15-19 who have begun childbearing ranges from 15% in Kigezi region to 28 % -30% in Busoga and Bukedi sub regions.
* The proportion of women age 15-19 who have begun childbearing decreases with both education and wealth.

Regions: The selection of the districts that we surveyed was informed by HIV prevalence dynamics and implementing partner support: we went to districts where there were Global Fund-supported implementing partners working to reduce the new number of new HIV infections among AGYW, improve SRH (e.g. reduce teenage pregnancy) and GBV indicators in the targeted districts.

## Data Processing

The output of this notebook includes four cleaned data frames, ready for further analysis.

## Load data

In [1]:
# Import libraries
import warnings
import os
import pandas as pd
import numpy as np
import re
import missingno as msno
import seaborn as sns
import matplotlib.pyplot as plt
from tabulate import tabulate
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Set environment
pd.options.display.float_format = '{:.2f}'.format
pd.set_option('display.max_colwidth', None)
sns.set_theme(style="whitegrid", context="paper")
os.chdir('/Users/nataschajademinnitt/Documents/5_data/teenage_pregnancy')
print("Current directory:", os.getcwd())
warnings.filterwarnings("ignore")

Current directory: /Users/nataschajademinnitt/Documents/5_data/teenage_pregnancy


## Data quality and description

In [10]:
# Load the Stata file (.dta)
df_load = pd.read_stata("./data/AGYW_dataset_2018.dta")



df_load = df_load.dropna(axis=1, how='all') # drop null columns
df_load = df_load.drop(columns=df_load.loc[:, 'Force_threat':'Useful_item'].columns)
df_load = df_load.drop(columns=['Continues', 'District_code'])
df_load = df_load.dropna(subset=['Been_preg'])

In [11]:
total = df_load['been_preg'].shape[0]
count = (df_load['been_preg'] == 1).sum()
print(f"The raw dataset contains responses from {total} girls (10-24) of which {count} have been pregnancy.")

KeyError: 'been_preg'

**Listing out column names:**

In [12]:
for i, c in enumerate(df_load.columns):
    print(i, c)

0 Uniquekey
1 Part_number
2 School_name
3 Date
4 Scol_status
5 Scol_own
6 Scol_location
7 Locate_other
8 Scol_level
9 Born_month
10 Age_completed
11 Born_year
12 Attend_scol
13 Level_scol
14 Left_scol
15 Lack_fees
16 Got_preg
17 Got_married
18 Got_sick
19 Need_money
20 Good_std
21 Int_scol
22 Specify_res
23 Other_reas
24 Return_scol
25 Like_return
26 Read_write
27 Able_read
28 Miss_scol
29 Who_live
30 Sick
31 Scol_fees
32 Had_period
33 Help_home
34 Bby_sit
35 Current_married
36 Earn_money
37 Hang_frie
38 Boy_place
39 Study_exam
40 More_wife
41 Other_spec
42 Res_other
43 Dont_kno
44 Part_age
45 Recstatus
46 Religion
47 Religion_other
48 Main_occup
49 Reli_other
50 Month_exp
51 Radio
52 Tv_set
53 Bicycle
54 Motorcycle
55 Own_home
56 Cell_phone
57 Reg_phone
58 Computer
59 Income_busin
60 Bath_room
61 Run_water
62 electricity
63 car
64 generator
65 solar
66 Pub_most
67 Most_puberty
68 Pub_second
69 Second_pub
70 Prefer_info
71 Prefer_other
72 Most_sexual
73 Sexual_other
74 Sexual_second
75

**Listing out object columns:**

In [5]:
obj_cols = df_load.columns[df_load.dtypes.eq('object')].tolist()

print("\n".join(obj_cols))

part_number
school_name
date
locate_other
specify_res
res_other
religion_other
reli_other
most_puberty
second_pub
prefer_other
sexual_other
second_sexual
pref_moresexual
most_other
other_secondint
other_sympto
siktreat_other
media_other
phone_other
google_other
list_avoid
other_specificrelate
pres_other
other_sexperson
other_recentsex
interv_code
vilage_name
class
other_obtmethod
other_preg
private_other
other_period
other_antena
other_deliv
other_notbuy
other_publplace
other_privatemd
other_sources
acces_othermethod
other_sought
other_hit
other_educ
other_toilet
village
school
agyw
distcode
partnum
sampleid
district
datecollected
fieldhiv
fieldsyphilis
cphlhiv
syphillis
rprtitres
dnapcr
district


## Recoding

Q.1001 Have you ever been pregnant and gave birth to a baby? [been_preg]

Q.401 Have you ever had any sexual intercourse in your life? (By this, I mean when a man or boy puts his penis in a woman or girl’s vagina) [life_sex]

Q.405 The first time you had sexual intercourse with someone; would you say you were willing, somewhat willing or not willing at all? Willing means you gave permission or said it was OK or that you did it because you wanted to and not because someone forced you to do it [will_sex]

In [ ]:
df_clean = df_load.copy()

# Been pregnant (1 = Yes, 0 = No) [1001]
df_clean['been_preg'] = df_clean['been_preg'].map({1.0: 1, 2.0: 0}).astype(int)

# Life sex (1 = Yes, 0 = No) [401]
df_clean['life_sex'] = df_clean['life_sex'].map({1.0: 1, 2.0: 0}).fillna(-1).astype(int) # fill na with -1 for now

# life_sex (correcting erorrs where life_sex is coded as -1)
df_clean.loc[(df_clean['been_preg'] == 1) & (df_clean['life_sex'] == -1), 'life_sex'] = 1
df_clean.loc[(df_clean['been_preg'] == 0) & (df_clean['life_sex'] == -1), 'life_sex'] = 0
df_clean.loc[(df_clean['been_preg'] == 1) & (df_clean['life_sex'] == 0), 'life_sex'] = 1

# Willingness to have sex [405]
df_clean['will_sex_binary'] = df_clean['will_sex'].map({1.0: 1, # very willing
                                                        2.0: 1, # somewhat willing
                                                        3.0: 0, # not willing at all
                                                        4.0: 0}) # don't know

# Did you or your partner do anything to prevent pregnancy? [406]
df_clean['do_anything_binary'] = df_clean['do_anything'].map({1.0: 1, # yes
                                                        2.0: 0, # no
                                                        3.0: np.nan}) # don't remember

# Were you under the influence of alcohol or drugs? [408]
df_clean['under_influe_binary'] = df_clean['under_influe'].map({1.0: 1, # yes
                                                        2.0: 0, # no
                                                        3.0: np.nan}) # don't remember

Q.403 Which person did you have sex with for the first time? [person_sex]

In [ ]:
# Recategorize
def classify_partner(x):
    if x == 1:
        return 'boyfriend'
    elif x == 2:
        return 'husband'
    elif x >= 3:
        return 'other'
    else:
        return np.nan

df_clean['person_sex_group'] = df_clean['person_sex'].apply(classify_partner)

# Convert it to a categorical variable
df_clean['person_sex_group'] = pd.Categorical(
    df_clean['person_sex_group'],
    categories=['boyfriend', 'husband', 'other'],
    ordered=False
)

# Check the distribution
df_clean['person_sex_group'].value_counts()

Q.409 When was the last time you had sexual intercourse? [last_sex]

Q.411 In the past 12 months, how often did you use condoms with all these partners? [often_usecondom]

Q.414 Thinking of THE LAST TIME you had intercourse with this partner, did you (or your partner) use a condom? [partuse_condom]

Q.415 Thinking of all the times you had intercourse with this partner IN THE LAST 12 MONTHS, would you say you used a condom all the time, sometimes, or never? [some_times]

Q.416b Some people are worried about pregnancy. How concerned are you or were you about getting pregnant? [worry_preg]

In [ ]:
# --- Inputs can be strings or numbers; coerce safely ---
last = pd.to_numeric(df_clean['last_sex'], errors='coerce')          # 1=≤1wk, 2=≤1mo, 3=>1–12mo, 4=>12mo
ocon = pd.to_numeric(df_clean['often_usecondom'], errors='coerce')

# 1) Activity indicator (used later to keep inactives without dropping them)
df_clean['sex_active_12m'] = ocon.isin([1,2,3]).astype(int)

# 2) Ordinal consistency among the active: Never=0, Sometimes=1, Always=2
cond_map = {1: 0, 2: 1, 3: 2}        # 4 and 98 become NaN
df_clean['condom_use_ord'] = ocon.map(cond_map)

# 3) Modeling-friendly composite:
#    - sets score to 0 for inactives, but keeps separate flags so effects are identifiable
df_clean['condom_use_ord_active'] = df_clean['condom_use_ord'].fillna(0) * df_clean['sex_active_12m']

# --- QC: quick sanity checks ---
print("Active last 12m (binary):")
print(df_clean['sex_active_12m'].value_counts(dropna=False))

print("\nCondom-use ordinal among active (0=never, 1=sometimes, 2=always):")
print(df_clean.loc[df_clean['sex_active_12m'].eq(1), 'condom_use_ord'].value_counts(dropna=False))

print("\nCross-check activity by last_sex code:")
print(pd.crosstab(last, df_clean['sex_active_12m'], rownames=['last_sex'], colnames=['active12m']))

In [ ]:
# Order condom use last 12 months as 1 (Never), 2 (Sometimes), 3 (Always) [414]
df_clean.loc[df_clean['some_times'].isin([4, 98]), 'some_times'] = np.nan
df_clean['some_times'] = pd.to_numeric(df_clean['some_times'])

# For pregnancy concern [416b]
df_clean.loc[df_clean['worry_preg'].isin([98]), 'worry_preg'] = np.nan
df_clean['worry_preg'] = pd.to_numeric(df_clean['worry_preg'])
# reverse code higher values indicate greater concern (range from 1 to 4)
df_clean['worry_preg_reverse'] = 5 - df_clean['worry_preg'] 

# For condom use at last sex [414]
df_clean['partuse_condom_binary'] = np.where(df_clean['partuse_condom'] == 1, 1, 0)

1003 How did the first pregnancy end? [preg_end]

1011 Have you ever had a abortion? [had_abort]

1015 Was your most recent abortion spontaneous or induced by you? [???]

Q.intro SCHOOLING STATUS: 1…… IN-SCHOOL 2 ……. OUT OF SCHOOL [scol_status]

Q.103a Have you ever attended school? [atttend_scol]

Q.104c What are some of the main reasons you are not in school? (Select all that apply)

In [ ]:
# If not in school and attend school is blank replace with never attended school [Intro Q]
df_clean.loc[(df_clean['scol_status'] == 'Out of school') & (df_clean['attend_scol'].isna()), 'attend_scol'] = 'No'

# Have you ever attended school? [103a]
df_clean['attend_scol_binary'] = df_clean['attend_scol'].map({'Yes':1,'No':0})

# Reasons for drop out [104c]
columns_to_encode = ['lack_fees', 'got_preg', 'got_married', 'got_sick', 'need_money', 'good_std', 'int_scol']
df_clean[columns_to_encode] = df_clean[columns_to_encode].fillna(0)
df_clean[columns_to_encode] = df_clean[columns_to_encode].map(lambda x: 1 if x == 1 else 0)

Q.104a What is the highest level of school you attained?

In [ ]:
# 1) Normalize level_scol to a canonical string
def norm_level_str(x):
    if pd.isna(x):
        return np.nan
    s = str(x).upper()

    # unify quotes/dashes
    s = (s.replace('’', "'").replace('‘', "'")
           .replace('–', '-').replace('—', '-'))

    # replace odd Unicode spaces with normal space
    s = (s.replace('\u00A0', ' ')   # NBSP
           .replace('\u202F', ' ')  # narrow NBSP
           .replace('\u2009', ' ')) # thin space

    s = s.strip()

    # standardize whitespace and hyphen spacing
    s = re.sub(r'\s+', ' ', s)
    s = re.sub(r'\s*-\s*', ' - ', s)

    # unify O/A LEVEL
    s = re.sub(r'\bO\s*LEVEL\b', "O' LEVEL", s)
    s = re.sub(r'\bA\s*LEVEL\b', "A' LEVEL", s)

    # standardize dotted tokens inside ranges
    s = re.sub(r'\bP\s*\.?\s*([1-7])\b', r'P.\1', s)
    s = re.sub(r'\bS\s*\.?\s*([1-6])\b', r'S.\1', s)

    # normalize the common range text
    s = re.sub(r'(P\.[1-7])\s*-\s*(P\.[1-7])', r'\1 - \2', s)
    s = re.sub(r'(S\.[1-6])\s*-\s*(S\.[1-6])', r'\1 - \2', s)

    # simple alias
    if s == 'OTHER':
        s = 'OTHER (SPECIFY)'

    return s

lvl_norm = df_clean['level_scol'].apply(norm_level_str)

# 2) First pass: exact matches to your canonical labels
codes = pd.Series(pd.NA, index=df_clean.index, dtype='Int64')

exact_map = {
    "PRIMARY (P.1 - P.7)": 2,
    "PRIMARY PROFESSIONAL": 3,
    "O' LEVEL (S.1 - S.4)": 4,
    "O' LEVEL PROFESSIONAL": 5,
    "A' LEVEL (S.5 - S.6)": 6,
    "UNIVERSITY": 7,
    "OTHER TERTIARY (AFTER S.6)": 8,
    "OTHER (SPECIFY)": 9,
}
for key, val in exact_map.items():
    codes[lvl_norm == key] = val

# 3) Fallback: regex categories for anything still unmapped
need = codes.isna()
ln = lvl_norm[need]

# university / tertiary
codes.loc[need & ln.str.contains(r'\b(UNIVERSITY|COLLEGE|TERTIARY|INSTITUTE|POLYTECH|DIPLOMA|DEGREE)\b', na=False)] = 8
codes.loc[need & ln.str.contains(r'\bUNIVERSITY\b', na=False)] = 7  # keep 7 for explicit "UNIVERSITY"

# A level / upper secondary (S5–S6, A' LEVEL, FORM 5–6)
codes.loc[need & ln.str.contains(r"\bA' LEVEL\b|\bS\.[5-6]\b|\bFORM\s*[5-6]\b", na=False)] = 6

# O level / lower secondary (S1–S4, O' LEVEL, FORM 1–4)
codes.loc[need & ln.str.contains(r"\bO' LEVEL\b|\bS\.[1-4]\b|\bFORM\s*[1-4]\b", na=False)] = 4
codes.loc[need & ln.str.contains(r"\bO' LEVEL\b.*PROF|PROFESSIONAL", na=False)] = 5

# Primary (P1–P7, PRIMARY, PRI)
codes.loc[need & ln.str.contains(r'\bPRIMARY\b|\bPRI\b|\bP\.[1-7]\b', na=False)] = 2
codes.loc[need & ln.str.contains(r'\bPRIMARY\b.*PROF|PROFESSIONAL', na=False)] = 3

# (Optional) None / never attended – only if you have explicit wording
# codes.loc[need & ln.str.contains(r'\b(NONE|NO SCHOOL|NEVER ATTENDED)\b', na=False)] = pd.NA

df_clean['level_scol_recode'] = codes

Q. 105 If in school, in which class are you?

In [ ]:
# Work on a copy of the raw column
s = df_clean['class'].astype(str)

# 1) Normalize text
s = (s.str.strip()
       .str.upper()
       .str.replace('’', "'", regex=False)
       .str.replace('–', '-', regex=False)
       .str.replace('—', '-', regex=False))

# Treat obvious missing tokens as NaN
s = s.replace({'': np.nan, 'NAN': np.nan, 'NA': np.nan, 'NONE': np.nan})

# Keep only alphanumerics (helps with things like "[3", "S.", etc.)
norm = s.str.replace(r'[^A-Z0-9]', '', regex=True)

# 2) Fix special cases and common variants
# University / other tertiary (also catch "UNIVERSITY88" / "OTHERTERTIARY89")
norm = norm.str.replace(r'^UNIVERSITY(?:88)?$', '88', regex=True)
norm = norm.str.replace(r'^OTHERTERTIARY(?:89)?$', '89', regex=True)

# University years like Y1, Y2 -> University (88)
norm = norm.str.replace(r'^Y\d+$', '88', regex=True)

# Junior forms (J1..J3) -> S1..S3 (common data entry variant)
norm = norm.str.replace(r'^J([1-3])$', r'S\1', regex=True)

# Typos where leading 'S' became '5': 53..56 -> S3..S6
norm = norm.str.replace(r'^5([1-6])$', r'S\1', regex=True)

# Lone 'S' (no number) is ambiguous -> NaN
norm = norm.mask(norm.eq('S'))

# 98 was a mis-code for OTHER TERTIARY (89)
norm = norm.str.replace(r'^98$', '89', regex=True)

# Bare digits 1–7 -> P1–P7 (fillna(False) to avoid the boolean mask error)
m_bare = norm.str.fullmatch(r'[1-7]').fillna(False)
norm.loc[m_bare] = 'P' + norm.loc[m_bare]

# 3) Accept only valid targets
valid = {*(f'P{i}' for i in range(1,8)),
         *(f'S{i}' for i in range(1,7)),
         '88', '89'}

# Pure digits other than 88/89 are ambiguous (e.g., '9') -> NaN
is_pure_digit = norm.str.fullmatch(r'\d+')
norm = norm.mask(is_pure_digit & ~norm.isin({'88','89'}))

class_clean = norm.where(norm.isin(valid))
df_clean['class_clean'] = pd.Categorical(class_clean, categories=[*(f'P{i}' for i in range(1,8)),
                                                                 *(f'S{i}' for i in range(1,7)),
                                                                 '88','89'])

print(df_clean['class_clean'].value_counts(dropna=False).head(20))

mapping = {
    "PRIMARY (P.1 - P.7)": 2,
    "PRIMARY PROFESSIONAL": 3,
    "O' LEVEL (S.1 - S.4)": 4,
    "O' LEVEL PROFESSIONAL": 5,
    "A' LEVEL (S.5 - S.6)": 6,
    "UNIVERSITY": 7,
    "OTHER TERTIARY (AFTER S.6)": 8,
    "OTHER (SPECIFY)": 9,
}

Q.107a Are you currently married? (by marriage I mean religious, traditional, civil or consensual union) [current_married]

In [ ]:
# Marital status to binary numeric (1 = Married, 0 = Never Married) [107a]
df_clean['current_married_binary'] = df_clean['current_married'].map({
    'MARRIED/UNION': 1,
    'DIVORCED/SEPARATED':0,
    'WIDOWED':0,
    'NEVER MARRIED': 0,
    'IN RELATIONSHIP BUT NOT MARRIED': 0
})

# Marital history to binary numeric (1 = Been married, 0 = Never Married) [107a]
df_clean['been_married_binary'] = df_clean['current_married'].map({
    'MARRIED/UNION': 1,
    'DIVORCED/SEPARATED':1,
    'WIDOWED':1,
    'NEVER MARRIED': 0,
    'IN RELATIONSHIP BUT NOT MARRIED': 0
})

Q.1302 How many times have you sought services or information from a doctor or a nurse for these services in the last twelve months? [numb_times]

In [ ]:
# Replace 0 with 1 since every individual is assumed to have visited at least once [1302]
df_clean['numb_times'] = df_clean['numb_times'].replace(0, 1)

# Compute the IQR to set an upper bound for outliers
q1 = df_clean['numb_times'].quantile(0.25)
q3 = df_clean['numb_times'].quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr

# Cap values above the upper bound to the upper bound value
df_clean['numb_times'] = df_clean['numb_times'].clip(upper=upper_bound)

Q.1101 How old were you when you first got married? [age_marry]

In [ ]:
# Compute the IQR for age_marry
q1 = df_clean['age_marry'].quantile(0.25)
q3 = df_clean['age_marry'].quantile(0.75)
iqr = q3 - q1

# Calculate the lower bound
lower_bound = q1 - 1.5 * iqr

# Cap values below the lower bound to the lower_bound value
df_clean['age_marry'] = df_clean['age_marry'].clip(lower=lower_bound)

# Married during adolesence
df_clean['married_by19'] = (df_clean['age_marry'] <= 19).astype(float)

Q.101 In what month and year were you born? [born_month]

Q.102 How old were you at your last birthday? [age_completed]

In [ ]:
# If born_year is missing, estimate it using age_completed
df_clean.loc[df_clean['born_year'].isna(), 'born_year'] = 2018 - df_clean['age_completed']

# If born_month is missing, assume July as the midpoint of the year
df_clean.loc[df_clean['born_month'].isna(), 'born_month'] = 7

# Ensuring numeric dtypes
df_clean['born_month'] = pd.to_numeric(df_clean['born_month'], errors='coerce')  # Convert to numeric, set errors to NaN
df_clean['born_year'] = pd.to_numeric(df_clean['born_year'], errors='coerce')  # Convert to numeric

# Fix `born_month`: Keep only values between 1 and 12, replace others with NaN
df_clean.loc[~df_clean['born_month'].between(1, 12), 'born_month'] = np.nan

# Fix `born_year`: Keep only reasonable values (between 1993 and 2008)
df_clean.loc[~df_clean['born_year'].between(1993, 2008), 'born_year'] = np.nan

# Assume missing `born_month` is July (middle of the year)
df_clean['born_month'] = df_clean['born_month'].fillna(7)

# Calculate exact months lived by July 2018
df_clean['age_months'] = (2018 - df_clean['born_year']) * 12 + (7 - df_clean['born_month'])

# Making sure that the pregnancy age is between 10 and 24
df_clean.loc[~df_clean['age_preg'].between(10, 24), 'age_preg'] = np.nan

# Missing age_preg is replaced with current age minus 1 year
df_clean.loc[(df_clean['been_preg'] == 1) & (df_clean['age_preg'].isna()), 'age_preg'] = df_clean['age_completed'] - 1

Q.402 How old were you when you had sexual intercourse for the very first time? [age_sex]

Q.1001 Have you ever been pregnant and gave birth to a baby? [been_preg]

Q.102 How old were you at your last birthday? [age_completed]

Q.1101 How old were you when you first got married? [age_marry]

Q.101 In what month and year were you born? [born_month]

In [ ]:
# 1. Update 'been_preg' based on 'age_preg'
df_clean.loc[(df_clean['been_preg'] == 0) & (df_clean['age_preg'].notna()), 'been_preg'] = 1

# 2. Validate 'sex_age'
df_clean.loc[~df_clean['sex_age'].between(5, 24), 'sex_age'] = np.nan

# 3. Assign 'sex_age' where missing for pregnant individuals
df_clean.loc[(df_clean['been_preg'] == 1) & (df_clean['sex_age'].isna()), 'sex_age'] = df_clean['age_preg']

# 4. Ensure 'sex_age' ≤ 'age_preg'
df_clean.loc[(df_clean['been_preg'] == 1) & (df_clean['sex_age'] > df_clean['age_preg']), 'sex_age'] = df_clean['age_preg']

# 5. Ensure 'age_preg' ≤ 'age_completed'
df_clean.loc[(df_clean['been_preg'] == 1) & (df_clean['age_preg'] > df_clean['age_completed']), 'age_preg'] = df_clean['age_completed']

# 6. Ensure that 'age_marry' ≤ 'age_completed'
df_clean.loc[(df_clean['age_marry'] > df_clean['age_completed']), 'age_marry'] = df_clean['age_completed']

# 7. Update 'age_marry' based on 'been_married'
df_clean.loc[(df_clean['been_married_binary'] == 0) & (df_clean['age_marry'].notna()), 'age_marry'] = np.nan

# 8. Remove invalid 'age_preg' entries
df_clean = df_clean[(df_clean['age_preg'].isna()) | (df_clean['age_preg'] <= 25)].copy()

# 9. Calculate 'diff'
df_clean.loc[df_clean['been_preg'] == 1, 'diff'] = (df_clean['age_months'] / 12) - df_clean['age_preg']

# 10. Handle negative 'diff' values
df_clean.loc[df_clean['diff'] < -0.5, 'diff'] = np.nan

# 11. Pregnant during adolesence
df_clean['ado_preg'] = np.where((df_clean['been_preg'] == 1) & (df_clean['age_preg'] <= 19), 1, 0).astype('int8')

### Sample sizes for different subsetting

1. Sample 1 undercounts pregnancy events by excluding women who got pregnant as teens but are now older.
2. Sample 2 includes all teenage pregnancies (good), but compares them to girls who are still in the risk window—some of whom might yet become pregnant (so there’s censoring).
3. Sample 3 compares all teenage pregnancies to women who have passed through the risk window without becoming pregnant. This eliminates right‑censoring and gives a non‑pregnant group whose “at‑risk” period is complete.

In [ ]:
# Time Sensitive
## Not been pregnant (currently 10-19) | Been pregnant (currently 10-19)
df_sample_1 = df_clean.loc[(df_clean['age_completed'] <= 19)]
df_sample_1.ado_preg.value_counts()

In [ ]:
# Time Insensitive
## Not been pregnant (all ages) | Been pregnant 10-19 (all ages)
df_sample_2 = df_clean.copy()

df_sample_2.ado_preg.value_counts()

In [ ]:
# Case: Did fall pregnant between 10-19 (irrespective of current age)
# Control: Did not fall pregnant during adolesence (now older than 19, outside risk window)

df_sample_3 = df_clean.loc[
    (df_clean['ado_preg'].eq(1)) |
    (df_clean['ado_preg'].eq(0) & df_clean['age_completed'].ge(20))
].copy()

# How many controls are never vs adult-first?
age_preg_num = pd.to_numeric(df_sample_3['age_preg'], errors='coerce')
n_never_ge20  = ((df_sample_3['been_preg'].eq(0)) & df_sample_3['age_completed'].ge(20)).sum()
n_adult_first = ((df_sample_3['been_preg'].eq(1)) & age_preg_num.ge(20)).sum()

print("Controls split — Never ≥20:", int(n_never_ge20), " | Adult-first ≥20:", int(n_adult_first))
print("Total controls (should equal ado_preg==0 in df_sample_3):",
      int((df_sample_3['ado_preg'].eq(0)).sum()))

In [ ]:
df_sample_3.ado_preg.value_counts()

## Creating categories

**Marriage timing with pregnancy**

* If both age_preg and age_marry are missing → never_marry_or_preg
* If age_preg present, age_marry missing → preg_never_marry
* If age_marry present, age_preg missing → marry_never_preg
* If both are present: age_marry > age_preg → marry_after_preg
* If both are present: age_marry ≤ age_preg → marry_before_preg (ties go to “before”)

In [ ]:
def marriage_any_vs_cases(row):
    ap = pd.to_numeric(row['age_preg'],  errors='coerce')
    am = pd.to_numeric(row['age_marry'], errors='coerce')

    if pd.isna(ap) and pd.isna(am):
        return 'never_marry_or_preg'
    if pd.notna(ap) and pd.isna(am):
        return 'preg_never_marry'
    if pd.isna(ap) and pd.notna(am):
        return 'marry_never_preg'
    # both present (ties → before)
    return 'marry_after_preg' if am > ap else 'marry_before_preg'

df_clean['marriage_timing'] = df_clean.apply(marriage_any_vs_cases, axis=1)

print(pd.crosstab(df_clean['marriage_timing'], df_clean['been_preg']))

**Highest level of education | School Complete**

In Uganda, Universal Primary Education (UPE) guarantees completion through P7, while Universal Secondary Education (USE) guarantees completion of lower secondary education up to senior four. Thus, completion of lower senior education marks the de facto end of free, guaranteed schooling. With this observation in mind, girls’ education was categorized as follows: 
* 0= in-school (either in primary or lower secondary education at the time of interview)
* 1= completed lower secondary education (O’ LEVEL (S.1 – S.4) 4) (age >= 18)
* 2= dropped out of school (i.e., did not complete any level of education)

Q.103a Have you ever attended school? [attend_scol]

Q.103b Are you still in school? [scol_status]

Q.104a What is the highest level of school you attained? [level_scol]

Q.105 If in school, in which class are you? ['class']

If 103b scol_status == "In school" ⇒ use 105 class_clean to infer highest achieved.

If 103b scol_status == "Out of school" ⇒ use 104a level_scol_recode.

If 103b scol_status is missing/odd ⇒ prefer 104a level_scol_recode, else fall back to class_clean.

And for “highest achieved” we use these rules:
* P1–P7 → less_than_UPE (P7 not yet completed)
* S1–S4 → UPE (lower sec not completed yet)
* S5–S6 → USE
* 88/89 → higher_than_USE

In [ ]:
# ---- Continuous years of schooling completed ----

# 1) Map CURRENT CLASS (for girls in school) to *years completed so far* (prev grade finished)
#    P1:0, P2:1, ..., P7:6, S1:7, S2:8, S3:9, S4:10, S5:11, S6:12
#    If currently at university/tertiary (88/89), minimum completed = A’Level (13 years).
years_from_class = {
    **{f'P{i}': i-1 for i in range(1,8)},          # P1→0 ... P7→6
    'S1': 7, 'S2': 8, 'S3': 9, 'S4': 10,           # lower secondary in progress
    'S5': 11, 'S6': 12,                            # upper secondary in progress
    '88': 13, '89': 13                             # in university/tertiary → at least 13 completed
}

cls = df_clean['class_clean'].astype('string').str.strip()
years_cls = cls.map(years_from_class).astype('float')

# 2) Map HIGHEST LEVEL COMPLETED (for girls out of school) to completed years
#    Matches your earlier convention (primary=7, O’level=11, A’level=13, tertiary+=14)
level = pd.to_numeric(df_clean['level_scol_recode'], errors='coerce')
years_from_level = pd.Series(np.nan, index=df_clean.index, dtype='float')
years_from_level[level.isin([2,3])] = 7    # completed primary (UPE)
years_from_level[level.isin([4,5])] = 11   # completed O'level (USE)
years_from_level[level.eq(6)]       = 13   # completed A'level
years_from_level[level.isin([7,8,9])] = 14 # completed tertiary+

# 3) Combine: take the maximum known “years completed”
years_school = pd.concat([years_cls, years_from_level], axis=1).max(axis=1)

# 4) Never attended / missing → 0
#    If you have an explicit “never attended” code, you can set those rows to 0 first.
years_school = years_school.fillna(0).clip(lower=0)

df_clean['years_school'] = years_school

In [ ]:
# --- 1) Map class_clean -> highest *achieved so far* (when currently in school) ---
# You already created df_clean['class_clean'] using your normalization chunk.
class_to_band = {
    **{f'P{i}': 'less_than_UPE'   for i in range(1,8)},  # P1–P7 (P7 not yet completed)
    **{f'S{i}': 'UPE'             for i in range(1,5)},  # S1–S4 (UPE completed; lower sec not completed)
    **{f'S{i}': 'USE'             for i in range(5,7)},  # S5–S6 (USE completed)
    '88': 'higher_than_USE',                           # University (while enrolled)
    '89': 'higher_than_USE',                           # Other tertiary (while enrolled)
}
from_class_band = df_clean['class_clean'].map(class_to_band)

# --- 2) Map level_scol_recode -> highest achieved (when out of school) ---
level_band_map = {
    2: 'UPE', 3: 'UPE',
    4: 'USE', 5: 'USE',
    6: 'higher_than_USE', 7: 'higher_than_USE',
    8: 'higher_than_USE', 9: 'higher_than_USE'
}
from_level_band = pd.to_numeric(df_clean['level_scol_recode'], errors='coerce').map(level_band_map)

# Optional: explicit "never attended" if you have attend_scol
if 'attend_scol' in df_clean.columns:
    attend_norm = df_clean['attend_scol'].astype(str).str.strip().str.lower()
    never_attended = attend_norm.isin(['no','0','false'])
else:
    never_attended = pd.Series(False, index=df_clean.index)

# --- 3) Combine per your precedence rules (status → class/level) ---
status = df_clean['scol_status'].astype(str).str.strip().str.lower()
is_in  = status.eq('in school')
is_out = status.eq('out of school')

edu_high = pd.Series(pd.NA, index=df_clean.index, dtype='object')

# In school: prefer class; fallback to level if missing
idx = is_in.fillna(False)
edu_high.loc[idx] = from_class_band.loc[idx]
miss = idx & edu_high.isna() & from_level_band.notna()
edu_high.loc[miss] = from_level_band.loc[miss]

# Out of school: prefer level; fallback to class if missing
idx = is_out.fillna(False)
edu_high.loc[idx] = from_level_band.loc[idx]
miss = idx & edu_high.isna() & from_class_band.notna()
edu_high.loc[miss] = from_class_band.loc[miss]

# Status missing/odd: prefer level; else class
idx = ~(is_in.fillna(False) | is_out.fillna(False))
edu_high.loc[idx] = from_level_band.loc[idx]
miss = idx & edu_high.isna() & from_class_band.notna()
edu_high.loc[miss] = from_class_band.loc[miss]

# Never attended overrides everything
edu_high.loc[never_attended] = 'None'

# Replace remaining NaNs with explicit 'Unknown' so denominators foot
cats = ['None','less_than_UPE','UPE','USE','higher_than_USE','Unknown']
edu_high = pd.Series(edu_high).fillna('Unknown')
df_clean['edu_bucket_highest'] = pd.Categorical(edu_high, categories=cats, ordered=True)

print("\nedu_bucket_highest (with 'Unknown'):")
print(df_clean['edu_bucket_highest'].value_counts(dropna=False))

# --- 4) Schooling status wrt USE (respect reported status; no under-18 override) ---
# 0) Helpers
def _norm_status(s):
    return (pd.Series(s, dtype="string")
              .str.strip().str.lower()
              .where(lambda x: x.isin(["in school","out of school"]), other=pd.NA))

# 1) Build 3-level completion status
#    0 = in-school (currently in P/S levels)
#    1 = completed lower secondary (USE) or higher (i.e., "school complete" for free, guaranteed schooling)
#    2 = dropped out (not in school AND has NOT completed lower secondary)
def make_school_complete3(df):
    status = _norm_status(df["scol_status"])
    ebh = df["edu_bucket_highest"].astype("string")  # expected: less_than_UPE, UPE, USE, higher_than_USE, Unknown/None

    out = pd.Series(pd.NA, index=df.index, dtype="Int64")
    # In school now
    out[status.eq("in school")] = 0
    # Out of school AND completed lower secondary or more
    out[status.eq("out of school") & ebh.isin(["USE","higher_than_USE"])] = 1
    # Out of school AND has not completed lower secondary
    out[status.eq("out of school") & ebh.isin(["less_than_UPE","UPE"])] = 2

    # If status missing, infer from highest achieved (keeps Unknown as NA)
    out[status.isna() & ebh.isin(["USE","higher_than_USE"])] = 1
    out[status.isna() & ebh.isin(["less_than_UPE","UPE"])] = 2

    return out.astype("Int64")

df_clean["school_complete3"] = make_school_complete3(df_clean)
label_map = {0:"in_school", 1:"completed_lower_secondary", 2:"dropped_out"}
df_clean["school_complete3_lbl"] = df_clean["school_complete3"].map(label_map).astype("string")

print("\nschooling_status_USE counts:")
print(df_clean['school_complete3_lbl'].value_counts(dropna=False))

**Age Cohort**

In [ ]:
df_clean['age_cohort'] = np.where(df_clean['age_completed'] <= 14, '10–14', '15–19')

**Household Vulnerability**

Vulnerability was measured at the individual, household and community levels, following the steps outlined in the report entitled, The Adolescent Girls Vulnerability Index: Guiding Strategic Investment in Uganda.

For household level, a girl was considered vulnerable if she experienced any two of the following five conditions: no access to improved source of water, no access to improved sanitation, household head has no education, food insecurity (no access to food in a day), and non-family support (ever consulted others for social support other than a family member).

In [ ]:
# Conditions met for HH vulnerability index
df_clean['pre_ques'] = df_clean.loc[:, 'sick_adult':'care_orphan'].apply(lambda row: (row == 1).sum(), axis=1)
df_clean['hh_vul'] = 0  # Default to 0 for all

# Only apply vulnerability conditions if pre_ques > 0
df_clean.loc[df_clean['pre_ques'] > 0, 'hh_vul'] = (
    (
        (df_clean['source_water'] == 3).astype(int) +  # No improved water
        (df_clean['type_toilet'] != 1).astype(int) +  # No improved sanitation
        (df_clean['highest_educ'].isna()).astype(int) +  # Household head has no education (skipped question)
        (df_clean['enough_food'] != 1).astype(int) +  # Food insecurity (at least 1 day without food)
        (df_clean['consult_spirit'] != 1).astype(int)  # Consulted non-family for social support
    ) >= 2  # If at least two conditions are met → Vulnerable (1), otherwise (0)
).astype(int)  # Ensure the column contains only 0s and 1s

# Tabulate population count
hh_vul_count = df_clean.groupby(['hh_vul', 'been_preg']).size().reset_index(name='count')
hh_vul_count['percentage'] = hh_vul_count['count'] / hh_vul_count.groupby('hh_vul')['count'].transform('sum') * 100
hh_vul_count

In [ ]:
# Wealth quintiles
asset_vars = ['radio',
              'tv_set',
              'bicycle',
              'motorcycle',
              'own_home',
              'cell_phone',
              'reg_phone',
              'computer',
              'income_busin',
              'bath_room',
              'run_water',
              'electricity',
              'car',
              'generator',
              'solar'
             ]

# Handle categorical responses: Replace '98' with NaN and convert to binary
df_assets = df_clean[asset_vars].replace(98, np.nan).copy()
df_assets = df_assets.map(lambda x: 1 if x == 1 else 0)

# Fill NaN with 0 (assuming lack of asset ownership if unsure)
df_assets = df_assets.fillna(0)

# Standardize the data
scaler = StandardScaler()
assets_scaled = scaler.fit_transform(df_assets)

# Apply PCA
pca = PCA(n_components=1)
wealth_index = pca.fit_transform(assets_scaled)

# Assign the first principal component as a flat array
df_clean['wealth_index'] = wealth_index[:, 0]

# Create wealth tertiles
df_clean['wealth_tertile'] = pd.qcut(df_clean['wealth_index'], 3, labels=['Low', 'Medium', 'High'])

df_clean['wealth_tertile'] = pd.Categorical(
    df_clean['wealth_tertile'],
    categories=['Low', 'Medium', 'High'],
    ordered=True
)

# Summary table
wealth_summary = df_clean['wealth_tertile'].value_counts().sort_index().reset_index()
wealth_summary.columns = ['Wealth Tertile', 'Count']
wealth_summary['Percentage'] = (wealth_summary['Count'] / wealth_summary['Count'].sum() * 100).round(1)

print(wealth_summary)

# Visualize the Wealth Tertiles
plt.figure(figsize=(8, 5))
sns.countplot(x='wealth_tertile', data=df_clean, palette="viridis", hue='wealth_tertile')
plt.xlabel("Wealth Tertile")
plt.ylabel("Number of Households")
plt.title("Distribution of Wealth Tertiles")
sns.despine(top=True, right=True)
plt.tight_layout()
plt.savefig('wealth_tertiles.png', dpi=300)
plt.show()

# Display PCA weights
wealth_weights = pd.Series(pca.components_[0], index=asset_vars)
print(wealth_weights.sort_values(ascending=False))

## Export

In [ ]:
FEATURES = [
    # outcome & sampling
    'ado_preg', 'been_preg', 'age_completed', 'age_cohort',

    # pregnancy & timing
    'age_preg', 'age_preg_year',

    # sexual debut & exposure
    'sex_age', 'sex_age_teen', 'risk_time', 'sex_age_band', 'initiated', 'will_sex_binary', 'do_anything_binary',

    # marriage
    'age_marry', 'marry_age_teen', 'married_by19', 'marriage_timing', 'marriage_timing_at_preg',

    # schooling
    'school_complete3_lbl', 'years_school', 'level_scol_recode', 'edu_bucket_highest',

    # partner at debut
    'person_sex_group', 'partner_missing',

    # contraception at debut
    'male_condom', 'female_condom', 'iud_coil', 'avoid_other', 'pill', 'withdrawal', 'implant',
    'condom_first',

    # condom frequency last 12 months
    'often_usecondom', 'sex_active_12m', 'condom_use_ord', 'condom_use_ord_active',
    'consistent_condom', 'sex_inactive_12m', 'condom_unknown_12m',

    # table helpers
    'method_group', 'sex_age_year', 'wealth_tertile'
]

cols = [c for c in FEATURES if c in df_clean.columns]
df_export = df_clean[cols].copy()

In [ ]:
# Export as csv
df_export.to_csv('./data/processed_df_2018.csv', index=False)
df_export.info()

In [ ]:
for c in df_export.columns:
    nunique = df_export[c].nunique(dropna=False)
    print(f"\n=== {c} (nunique={nunique}) ===")
    print(df_export[c].value_counts(dropna=False))
